# S3 - Is the famfull family-transfer real, or RNA-memorization? (confound gate)
**Question.** famfull shows held-out-FAMILY MCC 0.29-0.37 (S1). But are the RNAs held out? If a held (RNA, held-protein) pair can be scored from the RNA alone, the 'family transfer' is RNA-bindability memorization, not protein-family generalization.

*Data: `mmpartnet_out/famfull_confounds.json` (CPU, reusable) + `mmpartnet_out/famfull/`.*

## Definitions / math
- **RNA overlap:** fraction of held RNAs that also appear in train (should be LOW for a clean claim).
- **positive-RNA overlap:** fraction of held POSITIVE RNAs already positive (bound by some OTHER protein) in train.
- **RNA-only bindability baseline** (protein IGNORED): $\hat p(\text{bound}\mid r)=\dfrac{\#\{(r,\cdot)\in\text{train}: y=1\}}{\#\{(r,\cdot)\in\text{train}\}}$; score a held pair $(r,p)$ by $\hat p(\text{bound}\mid r)$ (threshold-swept).
- **MCC** $=\dfrac{TP\,TN-FP\,FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$.
- **Genuine protein signal (residual)** $=$ MCC(CORAL) $-$ MCC(RNA-only).

In [ ]:
import json; from pathlib import Path; from IPython.display import Markdown, display
OUT = Path('..')/'..'/'mmpartnet_out'
d = json.loads((OUT/'famfull_confounds.json').read_text())
tab='| fold | RNA overlap | pos-RNA overlap | prot overlap | RNA-only MCC | CORAL MCC | residual |\n'
tab+='|---|---|---|---|---|---|---|\n'
for f in d['per_fold']:
    res=f['coral_famfull_bestMCC']-f['rna_only_baseline_bestMCC']
    tab+=(f"| {f['fold']} | {f['rna_overlap_held_in_train']:.2f} | {f['pos_rna_overlap']:.2f} | "
          f"{f['protein_overlap']:.3f} | {f['rna_only_baseline_bestMCC']:.3f} | {f['coral_famfull_bestMCC']:.3f} | {res:+.3f} |\n")
display(Markdown(tab))
s=d['summary']
display(Markdown(f"**Mean: RNA-only MCC {s['rna_only_baseline_bestMCC_mean']:.3f} vs CORAL {s['coral_famfull_bestMCC_mean']:.3f} "
                 f"=> genuine protein residual ~{s['coral_famfull_bestMCC_mean']-s['rna_only_baseline_bestMCC_mean']:.3f} MCC. "
                 f"RNA overlap {s['rna_overlap_mean']:.2f}.**"))

In [ ]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt, numpy as np
fo=[f['fold'] for f in d['per_fold']]; rna=[f['rna_only_baseline_bestMCC'] for f in d['per_fold']]
cor=[f['coral_famfull_bestMCC'] for f in d['per_fold']]
x=np.arange(len(fo)); w=0.38; fig,ax=plt.subplots(figsize=(7,4))
ax.bar(x-w/2, rna, w, label='RNA-only baseline (protein ignored)', color='#c0504d')
ax.bar(x+w/2, cor, w, label='CORAL famfull (RNA+protein)', color='#4472c4')
ax.axhline(0.10, ls='--', c='gray', lw=1, label='famscale N<=200 (near-chance)')
ax.set_xticks(x); ax.set_xticklabels([f'fold {k}' for k in fo]); ax.set_ylabel('best held-out-family MCC')
ax.set_title('Family transfer vs RNA-memorization confound'); ax.legend(fontsize=8); fig.tight_layout()
fig.savefig('S3_confound.png', dpi=110); plt.close(fig)
from IPython.display import Image; display(Image('S3_confound.png'))

## Conclusion
The held **RNAs are ~100% shared** train<->held (only proteins are held out), and an **RNA-only bindability baseline reaches MCC ~0.23** (protein never seen) vs CORAL's 0.37. So **most of the famfull signal, and most of the famfull>famscale gain, is RNA-coverage memorization**, not protein-family transfer: more training families -> more training RNAs -> held RNAs better covered. The **genuine protein-family signal is the residual ~0.13 MCC** (best-epoch), highly fold-variable (~0 in folds 0/4, ~0.2 in folds 1/3).

**This does NOT confirm the interpolation hypothesis.** Two clean tests are needed (see plan + `S1_depeek_headline`, `S2_null_battery`): (1) **de-peek** the headline (report TEST at the val-selected epoch, not best-on-test); (2) a **protein-shuffle** control + an **RNA-AND-family-disjoint** split, to isolate the protein-transfer component. Until then, 'diversity is the lever' is **confounded by RNA coverage**.

*Provenance: `mmpartnet_out/famfull_confounds.json` (repo `scripts/famfull_confounds.py`, CPU, CORAL fork data).*